这两组方法分别解决了 GPU 编程中两个非常核心的问题：<font color='red'>“如何精确测量时间” </font>和 <font color='red'>“如何保证实验结果可复现”</font>。下面为你详细解释并举例：

## ⏱️ torch.cuda.synchronize(device=None)：解决异步执行的“时间差”
核心解释：
- <font color='red'>PyTorch 默认采用异步执行模型。</font>
  - <font color='red'>当你在代码中写一个 GPU 操作（比如矩阵乘法）时，CPU 只是把这个计算任务（Kernel）丢到 GPU 的任务队列里，然后就立刻跑去执行下一行代码了，并不会等待 GPU 真正算完。</font>
- 这就导致了一个问题：如果你想用 Python 的 time.time() 来测量一段 GPU 代码的运行时间，测出来的往往只是 CPU “下发指令”的时间，而不是 GPU 真正的“计算时间”，结果会严重偏小。
- torch.cuda.synchronize() 的作用就是<font color='red'>强制让 CPU 停下来等待，直到指定设备上的所有 CUDA 计算任务全部完成，再继续执行后面的代码。</font>

In [ ]:
import torch
import time

if torch.cuda.is_available():
    device = torch.device("cuda")
    # 创建两个较大的张量用于测试
    a = torch.randn(5000, 5000, device=device)
    b = torch.randn(5000, 5000, device=device)

    # --- 错误的时间测量方式（没有同步） ---
    start = time.time()
    c = torch.matmul(a, b)  # CPU 只是下发指令，不等 GPU 算完就跑了
    # 此时 GPU 还在后台默默计算，但代码已经跑到这里了
    end = time.time()
    print(f"未同步的虚假耗时: {(end - start) * 1000:.2f} ms") 

    # --- 正确的时间测量方式（加上同步） ---
    torch.cuda.synchronize() # 先清空之前残留的任务（可选但推荐）
    start = time.time()
    c = torch.matmul(a, b)
    torch.cuda.synchronize() # 关键！强制 CPU 等待 GPU 把上面的乘法彻底算完
    end = time.time()
    print(f"同步后的真实耗时: {(end - start) * 1000:.2f} ms")

## ⚠️ 避坑指南：
- 在正常的模型训练或推理循环中，千万不要在每一步都加 synchronize()，这会强制破坏 CPU 和 GPU 的并行工作，严重拖慢整体速度。<font color='red'>它通常只用在性能测试（Benchmark）或者调试时。</font>

## 🎲 torch.cuda.manual_seed() / manual_seed_all()：<font color='red'>保证实验可复现</font>

###### 核心解释：
<font color='green'>深度学习模型在初始化权重、数据打乱（Shuffle）或使用 Dropout 等层时，都会用到随机数。如果不固定随机数种子，你每次运行代码得到的初始权重和数据处理顺序都会不同，导致最终的结果（比如准确率）忽高忽低，无法复现。</font>
- torch.cuda.manual_seed(seed)：为当前使用的单张 GPU 设置随机数种子。
- torch.cuda.manual_seed_all(seed)：如果你的程序使用了多张 GPU（多卡并行），使用这个方法可以一次性为所有 GPU 设置相同的随机数种子，确保多卡环境下的结果也能复现。


In [ ]:
import torch

if torch.cuda.is_available():
    seed = 42  # 任意指定一个整数作为种子
    
    # 如果是单卡训练
    torch.cuda.manual_seed(seed)
    
    # 如果是多卡训练，推荐使用 manual_seed_all
    torch.cuda.manual_seed_all(seed)
    
    # 为了最严格的复现性，通常还会把 CPU 的种子也一起设了
    torch.manual_seed(seed)

    # 验证：连续运行两次这段代码，生成的随机张量数值会完全一样
    random_tensor = torch.randn(3, 3, device="cuda")
    print("固定种子后生成的随机张量：")
    print(random_tensor)

### 💡 最佳实践：
- 通常我们会把设置种子的代码放在整个 Python 脚本的最开头，作为实验环境的“定海神针”。